# Universal portfolio - Cover 1996

This is an example of how to evaluate a Universal portfolio. For more details see the library documentation 
https://azapy.readthedocs.io/en/latest/.

We start by importing **azapy** and other useful packages
(**azapy** version must be 1.2.1 or greater).

In [1]:
import azapy as az

print(f"azapy version {az.version()} >= 1.2.0")

azapy version 1.2.5 >= 1.2.0


### Collect historical market data

- `symb` is the list of stock symbols (portfolio components).
- `sdate` and `edate` are the start and end dates of historical time-series.
- `mktdir` is the name of the directory used as a buffer for market data collected from the data provider (in this case _alphavantage_).
    
> Note: if the flag `force=False` then a reading from `dir=mktdir` is attempted. If it fails, then the data provider servers will be accessed. The new data will be saved to the `dir=mktdir`. For more information see the readMkT documentation https://azapy.readthedocs.io/en/latest/.

In [2]:
mktdir = '../MkTdata'
sdate = '2012-01-01'
edate = '2023-06-30'
symb = ['VGT', 'SPY', 'XLV', 'GLD', 'ONEQ']

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir)

read VGT data from file
read SPY data from file
read XLV data from file
read GLD data from file
read ONEQ data from file

Request between 2012-01-03 : 2023-06-30
                    VGT         SPY         XLV         GLD        ONEQ
source            yahoo       yahoo       yahoo       yahoo       yahoo
force             False       False       False       False       False
save               True        True        True        True        True
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata
file_format         csv         csv         csv         csv         csv
api_key            None        None        None        None        None
nrow               2892        2892        2892        2892        2892
sdate        2012-01-03  2012-01-03  2012-01-03  2012-01-03  2012-01-03
edate        2023-06-30  2023-06-30  2023-06-30  2023-06-30  2023-06-30
error                No          No          No          No          No
extraction time 0.191 s


### Setup the Universal portfolio

First line is the constructor
Second line sets the model and execute the core computations. It is a Monte Carlo based evaluation.

- `mc_paths` - are the number of simulations per batch (must be >= 1)
- `nr_batches` - are the number of batches (must be >= 1). Note that the computation is multithreaded with one batch per thread.
- `variance_reduction = True` - default value (the MC will use the antithetic variance reduce implied by the permutations of the basket components)
- `alpha_dirichlet = None` - default value. In this case a uniform random generator of vectors in the M-simplex is used. This is equivalent to a Flat Dirichlet random generator (all alpha set to 1).

>Note the the total number of MC simulations is `mc_paths * nr_batches * M!` where `M` is the number of portfolio components and `M!` its factorial.

In [3]:
p4 = az.Port_Universal(mktdata, pname='UnivPort')    
port4 = p4.set_model(mc_paths=100, nr_batches=20, verbose=True)   

nr simulations: 240000
simulation time: 0.445903


### Historical portfolio weights

>Note: if `_CASH_` is not an explicit component of the portfolio (it is not present in the `makdata`), then its weight is
set to `0`.
A `_CASH_` asset can be added to the `mktdata` by using the helper function `azapy.add_cash_security(mktdata)`.

In [4]:
p4.get_weights()

,Droll,Dfix,GLD,ONEQ,SPY,VGT,XLV,_CASH_
0,2015-06-25,2015-06-24,0.200000,0.200000,0.200000,0.200000,0.200000,0
1,2015-09-25,2015-09-24,0.201676,0.199706,0.199552,0.199842,0.199224,0
2,2015-12-28,2015-12-24,0.198138,0.200509,0.200480,0.201178,0.199695,0
3,2016-03-28,2016-03-24,0.202431,0.198820,0.200343,0.200796,0.197610,0
4,2016-06-27,2016-06-24,0.204538,0.198037,0.199908,0.199482,0.198035,0
5,2016-09-27,2016-09-26,0.202581,0.199305,0.199469,0.201428,0.197216,0
6,2016-12-27,2016-12-23,0.197670,0.201062,0.201748,0.203129,0.196390,0
7,2017-03-28,2017-03-27,0.198569,0.200875,0.200573,0.203626,0.196357,0
8,2017-06-27,2017-06-26,0.196613,0.201420,0.200346,0.204361,0.197260,0
9,2017-09-26,2017-09-25,0.197298,0.201138,0.200289,0.204468,0.196808,0


### Portfolio performance view

In [5]:
_ = p4.port_view(fancy=True)

### Portfolio and its components relative performances 

In [6]:
_ = p4.port_view_all(fancy=True)

## Portfolio performance in terms of total return, maximum drawdown, and RoMaD 

- `RR` - total rate of return
- `DD` - maximum drawdown
- `RoMaD` - return over maximum drawdown (`RR/DD`)
- `DD_date` - maximum drawdown date
- `DD_start` - maximum drawdate stating date
- `DD_end` - maximum drawdown ending date

In [7]:
p4.port_perf(fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
UnivPort,13.19,-25.95,0.508337,2020-03-23,2020-02-19,2020-05-11,82
VGT,19.83,-35.07,0.565387,2022-10-14,2021-12-27,NaN,550
XLV,14.16,-28.40,0.498605,2020-03-23,2020-01-22,2020-07-15,175
ONEQ,16.63,-35.23,0.472037,2022-12-28,2021-11-19,NaN,588
SPY,13.58,-33.72,0.402723,2020-03-23,2020-02-19,2020-08-10,173
GLD,1.17,-42.11,0.027880,2015-12-17,2012-10-04,2020-07-22,2848


### Portfolio drawdowns (default - the first 5 largest)

- `DD` - value of the drawdown (percent)
- `Date` - drawdown date
- `Start` - drawdown starting date
- `End` - drawdown ending date

In [8]:
p4.port_drawdown(fancy=True)

,DD,Date,Start,End,NrDays
No,,,,,
1,-25.95,2020-03-23,2020-02-19,2020-05-11,82
2,-23.10,2022-10-14,2021-12-27,NaN,550
3,-15.54,2018-12-24,2018-10-03,2019-02-04,124
4,-11.02,2015-09-29,2015-07-17,2016-04-19,277
5,-9.37,2020-09-23,2020-09-02,2020-11-16,75


### Portfolio annual (calendar) rate of returns

>Note: the first and last year may not be a full calendar year.

In [9]:
p4.port_annual_returns(fancy=True)

,UnivPort
year,
2015,-3.59%
2016,8.95%
2017,24.24%
2018,4.60%
2019,27.41%
2020,31.60%
2021,19.45%
2022,-17.71%
2023,19.90%


### Portfolio monthly (clandar) rate of returns

In [10]:
p4.port_monthly_returns(fancy=True)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023
month,,,,,,,,,
1,nan%,-4.28%,3.66%,6.07%,6.77%,1.52%,-0.52%,-6.18%,6.20%
2,nan%,1.91%,4.44%,-2.37%,3.04%,-5.72%,-0.47%,-1.16%,-2.48%
3,nan%,4.92%,0.78%,0.19%,-1.45%,-2.85%,2.61%,4.48%,6.55%
4,nan%,0.37%,1.76%,0.18%,2.55%,12.60%,4.79%,-8.37%,1.03%
5,nan%,1.28%,1.85%,2.84%,-5.00%,5.16%,1.29%,-1.03%,2.04%
6,-1.88%,-0.43%,-1.30%,-2.53%,6.22%,-0.25%,1.19%,-6.90%,5.40%
7,0.72%,4.97%,2.56%,2.60%,1.18%,6.92%,2.90%,7.26%,nan%
8,-4.72%,-0.53%,2.14%,4.16%,0.06%,6.22%,2.61%,-4.66%,nan%
9,-3.77%,3.04%,0.91%,0.33%,0.90%,-3.43%,-5.78%,-7.24%,nan%


### Portfolio returns per reinvestment period 

- `Droll` - rolling date (rebalansing is assumed to be at the closing of the rolling date)
- `Dfinx` - fixing date (the most recent historical date included in the weights computations - last closing date)
- `RR` - portfolio rate of return on the rolling period starting on `Droll`
- *rest of the columns* - prevailing portfolio weights 

In [11]:
p4.port_period_returns(fancy=True)

,Droll,Dfix,RR,VGT,SPY,XLV,GLD,ONEQ
0,2015-06-25,2015-06-24,-7.77,20.00,20.00,20.00,20.00,20.00
1,2015-09-25,2015-09-24,4.15,19.98,19.96,19.92,20.17,19.97
2,2015-12-28,2015-12-24,0.70,20.12,20.05,19.97,19.81,20.05
3,2016-03-28,2016-03-24,0.64,20.08,20.03,19.76,20.24,19.88
4,2016-06-27,2016-06-24,7.94,19.95,19.99,19.80,20.45,19.80
5,2016-09-27,2016-09-26,0.96,20.14,19.95,19.72,20.26,19.93
6,2016-12-27,2016-12-23,7.45,20.31,20.17,19.64,19.77,20.11
7,2017-03-28,2017-03-27,4.16,20.36,20.06,19.64,19.86,20.09
8,2017-06-27,2017-06-26,2.64,20.44,20.03,19.73,19.66,20.14
9,2017-09-26,2017-09-25,6.53,20.45,20.03,19.68,19.73,20.11


### Number of shares per portfolio component for each rolling period

In [12]:
p4.get_nshares()

,GLD,ONEQ,SPY,VGT,XLV,_CASH_
Droll,,,,,,
2015-06-25,178,994,95,182,264,0
2015-09-25,169,996,96,183,268,0
2015-12-28,187,981,95,179,268,0
2016-03-28,169,1032,96,182,285,0
2016-06-27,158,1040,96,186,279,0
2016-09-27,170,1031,100,181,292,0
2016-12-27,194,989,94,174,299,0
2017-03-28,189,995,98,173,300,0
2017-06-27,197,972,98,168,291,0


### Other accounting informations

- `Droll` - the start of the rolling period (the end is the next period starting date)
- *portfolio symbols* - number of shares per portfolio component
- `cash_invst` - Dollar equivalent of the shares on the `Droll` date
- `cash_roll` - amount of uninvested cash rolled to the next period. A negative value indicates that the investor need to add this cash amount in order to execute the rolling. The main reasons for these small cash amounts are shares price differential between the fixing (computations) and rolling (execution) dates, as well as rounding to an integer the number of shares.
- `cash_divd` - amount of cash collected form dividend payments during the rolling period (it is an approximation considering the ex-dividend day as the dividend pay day - in practice between these dates could be a gap of a few days or even few weeks).

Note: the value of `cash_roll` can be minimized if,
1. set fixing date to be the same as the rolling date (implies the ability to run the portfolio optimization and execute the rolling transactions close to the end of trading day)
2. a large initial capital will lower, in a relative bases, the impact of rounding to an integer the number of shares.

In [13]:
p4.get_account(fancy=True)

,GLD,ONEQ,SPY,VGT,XLV,_CASH_,cash_invst,cash_roll,cash_divd
Droll,,,,,,,,,
2015-06-25,178,994,95,182,264,0.0,99923.14,76.86,0.00
2015-09-25,169,996,96,183,268,0.0,91764.49,863.18,396.01
2015-12-28,187,981,95,179,268,0.0,96884.54,232.27,271.30
2016-03-28,169,1032,96,182,285,0.0,97180.90,-1.86,289.09
2016-06-27,158,1040,96,186,279,0.0,96024.30,1553.60,304.14
2016-09-27,170,1031,100,181,292,0.0,107541.68,-665.27,321.15
2016-12-27,194,989,94,174,299,0.0,105975.03,-254.85,374.57
2017-03-28,189,995,98,173,300,0.0,114330.33,-402.24,276.80
2017-06-27,197,972,98,168,291,0.0,117613.88,901.94,271.02


# Universal Portfolio - weights evaluation on a fixing date

This is an example of how to evaluate efficiently the portfolio weights in a fixing date.
It follow a similar procedure as any other portfolio weights evaluation.

We will reuse the `mktdata` previously collected in cell [3].

We star by setting the universal portfolio from a fixing schedule. Later will do a similar computation where the universal portfolio will be set from a hard given fixing date.

### Set a fixing schedule 

For Universal portfolio, the computations of the weights in the fixing date requires the knowledge of all the previous 
fixing dates (as well as market data on these dates). We accomplish this by using `azapy.schedule_simple` function.

- `sdate` - start date of available historical data. The schedule start date will be greater but as close is possible to this date. 
- `edate`- end date of available historical data. The schedule end date is smaller but as close is possible to this date.
- `freq` - fixing frequency. It could be `M` for monthly or `Q` for quarterly.
- `noffset` - rolling date offset - number of business days offset form the last business day of the period (monthly or quarterly).
- `fixoffset` - fixing date offset in business days relative to the rolling date



In [14]:
fixing_schedule = az.schedule_simple(sdate, edate, 
                                     freq='M', noffset=-5, fixoffset=0)
fixing_schedule

,Droll,Dfix
0,2012-01-24,2012-01-24
1,2012-02-22,2012-02-22
2,2012-03-23,2012-03-23
3,2012-04-23,2012-04-23
4,2012-05-23,2012-05-23
...,...,...
134,2023-03-24,2023-03-24
135,2023-04-21,2023-04-21
136,2023-05-23,2023-05-23
137,2023-06-23,2023-06-23


## Set the `azapy.UniversalEngine` class from a fixing schedule

Alternatively, the UniversalEngine object can be set from a hard given fixing date (we will use this approach later in this script).

In [15]:
puniv = az.UniversalEngine(mktdata, schedule=fixing_schedule)

### Compute the portfolio weights

- `mc_paths` - are the number of simulations per batch (must be >= 1)
- `nr_batches` - are the number of batches (must be >= 1). Note that the computation is multithreaded with one batch per thread.
- `variance_reduction = True` - default value (the MC will use the antithetic variance reduce implied by the permutations of the basket components)
- `alpha_dirichlet = None` - default value. In this case a uniform random generator of vectors in the M-simplex is used. This is equivalent to a Flat Dirichlet random generator (all alpha set to 1).
- `mc_seed` - random generator seed value
- `verbose` - print out
    * number of MC simulations (`mc_paths * nr_batches * M!` if `variance_reduction = True` and `mc_paths * nr_batches` otherwise, where `M` is the number of portfolio components and `M!` its factorial).
    * simulation time
    
>Note: if `variance_reduction = True` (the default value) then both the effective number of MC simulations and the computation time grow factorial with the number of portfolio components. For large portfolios it is recommended to the impact of this setup.

>Note: multithreading under Python implementation has reduced effect. In our case we get a time reduction around 30%
   


In [16]:

ww = puniv.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, finalonly=False, verbose=True)

nr simulations: 192000
simulation time: 0.395941


### New portfolio weights

These are in the last raw of `ww`.

>Note: `ww` includes all historical weights (along the fixing schedule) - `finalonly` was set to `False`

In [17]:
ww

symbol,GLD,ONEQ,SPY,VGT,XLV
date,,,,,
2012-01-24,0.200000,0.200000,0.200000,0.200000,0.200000
2012-02-22,0.200701,0.200245,0.199646,0.200631,0.198777
2012-03-23,0.197935,0.201195,0.200116,0.201778,0.198975
2012-04-23,0.198036,0.200804,0.200038,0.201172,0.199949
2012-05-23,0.197724,0.200750,0.200205,0.200843,0.200478
...,...,...,...,...,...
2023-02-21,0.166779,0.207962,0.200435,0.218089,0.206735
2023-03-24,0.167973,0.208123,0.199412,0.219334,0.205159
2023-04-21,0.167363,0.207843,0.199840,0.218890,0.206065


## Set the `azapy.UniversalEngine` class 

### Build the UniversalEngine object

The fixing schedule is build internally going backward from the fixing date with a step of either 21 (for `freq='M'`) or 63 (for `freq='Q'`). The resulting schedule is approximatively equivalent with the one returned by `azapy.shedule_simple` function (that is using a business calendar and in general is more sophisticated).

In [18]:
puniv2 = az.UniversalEngine(mktdata, freq='M')

###  Weights computation - same as before

In [19]:
ww = puniv2.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, verbose=True)
print(ww)

nr simulations: 192000
simulation time: 0.432627
symbol
GLD     0.163962
ONEQ    0.210017
SPY     0.200289
VGT     0.222381
XLV     0.203350
Name: 2023-06-30 00:00:00, dtype: float64


## Set the `azapy.UniversalEngine` class with a Dirichlet random generator

We start by defining the Dirichlet alpha coefficient (must be between 0 and 1)

Here we choose all to be equal to the inverse of the number of portfolio components

In [20]:
dirichlet_alpha = [1 / len(symb)] * len(symb)

### Computation of the weights 

We start by passing the `dirichlet_alpha` coefficients to the constructor.

The rest is the same a above.

>Note: setting all alpha to 1 is equivalent to using a uniform random generator of vectors in the M-simplex of portfolio weights.

In [21]:
puniv2 = az.UniversalEngine(mktdata, freq='M', dirichlet_alpha=dirichlet_alpha)
ww = puniv2.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, verbose=True)
print(ww)

nr simulations: 192000
simulation time: 0.429335
symbol
GLD     0.109668
ONEQ    0.224763
SPY     0.195886
VGT     0.265875
XLV     0.203808
Name: 2023-06-30 00:00:00, dtype: float64
